In [27]:
def collect_all_paths(folder, n_threads=2, FILTER_KEYS=[]):
    partitions = folder.list_partitions()
    if FILTER_KEYS:
        filtered_partitions = [
            p for p in partitions
            if any(key in p for key in FILTER_KEYS)
            and "silver|" not in p
        ]
    else:
        filtered_partitions = partitions

    def get_paths(partition):
        info = folder.get_partition_info(partition)
        return info.get("paths", [])

    results = Parallel(
        n_jobs=n_threads,
        backend="threading",
        prefer="threads",
    )(
        delayed(get_paths)(p) for p in filtered_partitions
    )

    # Flatten list of lists
    all_paths = [path for sublist in results for path in sublist]

    return all_paths

In [28]:
def migrate_one_file(old_pd, new_pd, force, p):
    if "category=" in p:
        return {
            "ok": True,
            "dst": p,
            "attempts": 0,
            "skipped": True,
        }
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            # Rearrange path
            if "/raw/" in p:
                parts = p.replace("/raw/", "")
            else:
                parts = p.lstrip("/")
            parts = parts.split("/")
            try:
                instance_name = parts[0]
                category = parts[1]
                module = parts[2]
                year = parts[3]
                month = parts[4]
                day = parts[5]
                file_name = parts[6]
            except Exception as e:
                return {
                    "ok": False,
                    "path": p,
                    "error": str(e),
                    "attempts": attempt,
                }
            new_path = (
                f"raw/"
                f"category={category}/"
                f"module={module}/"
                f"instance_name={instance_name}/"
                f"year={year}/"
                f"month={month}/"
                f"day={day}/"
                f"{file_name}"
            )
            
            # Check if already exists
            if not force:
                try:
                    exists = new_pd.get_path_details(new_path).get("exists", False)
                except Exception:
                    exists = False  # fall back to attempting upload
                if exists:
                    return {
                        "ok": True,
                        "dst": new_path,
                        "attempts": 0,
                        "skipped": True,
                    }

            # Read old parquet
            with old_pd.get_download_stream(p) as stream:
                file_bytes = io.BytesIO(stream.read())
            df = pd.read_parquet(file_bytes)
            
            # add in timestamp for missing df for operating system stuff
            if category == "operating_system" and (
                        "timestamp" not in df.columns or df["timestamp"].isna().all()
                    ):
                date = f"{year}/{month}/{day}"
                df["timestamp"] = pd.to_datetime(date, format="%Y/%m/%d", utc=True)

            # Write to new location
            buffer = io.BytesIO()
            df.to_parquet(
                buffer,
                compression="gzip",
                engine="pyarrow",
                index=False,
            )
            buffer.seek(0)
            content = buffer.read()
            new_pd.upload_stream(new_path, content)

            return {
                "ok": True,
                "dst": new_path,
                "attempts": attempt,
            }
        
        except Exception as e:
            if attempt == MAX_RETRIES:
                return {
                    "ok": False,
                    "path": p,
                    "error": str(e),
                    "attempts": attempt,
                }

            # Exponential backoff + jitter
            sleep_time = (2 ** attempt) + random.random()
            time.sleep(sleep_time)
    return

In [29]:
# --------------------------------------------------
# MAIN
# --------------------------------------------------
import io
import os
import time
import pandas as pd
import dataiku
from joblib import Parallel, delayed
import time
import random


old_pd = dataiku.Folder("MQqfai90") # Old Folder
new_pd = dataiku.Folder("uBZVl2aJ") # New Folder

MAX_RETRIES = 3
N_THREADS =  os.cpu_count() - 1

start = time.time()
print(f"[INFO] Threads to be used: {N_THREADS}")

# --------------------------------------------------
# Collect partitions & total file count
# --------------------------------------------------
FILTER_KEYS = []
#FILTER_KEYS = ["|diskspace|", "|filesystem|"]
all_paths = collect_all_paths(old_pd, n_threads=N_THREADS, FILTER_KEYS=FILTER_KEYS)

print(f"[INFO] Total files to migrate: {len(all_paths)}")

# --------------------------------------------------
# Migrate the data
# --------------------------------------------------
results = Parallel(
    n_jobs=N_THREADS,
    backend="threading",
    prefer="threads",
)(
    delayed(migrate_one_file)(old_pd, new_pd, True, p)
    for p in all_paths
)

elapsed = time.time() - start

skipped = sum(1 for r in results if r.get("skipped"))
success = sum(1 for r in results if r.get("ok") and not r.get("skipped"))
failed = sum(1 for r in results if not r.get("ok"))

print("[DONE]")
print(f"[SUMMARY] Migrated....: {success}/{len(all_paths)}")
print(f"[SUMMARY] Skipped.....: {skipped}/{len(all_paths)}")
print(f"[SUMMARY] Failed......: {failed}/{len(all_paths)}")
print(f"[SUMMARY] Time: {elapsed/60:.1f} min")
print(f"[SUMMARY] Rate: {success/elapsed:.1f} files/sec")

[INFO] Threads to be used: 7
[INFO] Total files to migrate: 5909
[DONE]
[SUMMARY] Migrated....: 5909/5909
[SUMMARY] Skipped.....: 0/5909
[SUMMARY] Failed......: 0/5909
[SUMMARY] Time: 4.0 min
[SUMMARY] Rate: 24.7 files/sec
